# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR² dataset using the `mlcroissant` library. All references to record sets, fields, and columns use their Croissant `@id` identifiers to align with best practices in reproducible metadata-driven data science.

### Dataset Source
This dataset is described by a Croissant schema at the following URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Make sure mlcroissant is available (no-op if already installed)
!pip install -q mlcroissant

## 1. Data Loading
Load the dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the URL of the Croissant schema
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset via Croissant
dataset = mlc.Dataset(croissant_url)

# Access the metadata object
metadata = dataset.metadata
print(f"Dataset: {metadata.name}\n\n{metadata.description}")


## 2. Data Overview
Let's explore what record sets and fields are available to work with in this dataset.

We'll print out all available record set `@id` values and for each, their fields and corresponding `@id`s.

In [ ]:
# Explore record sets and fields by their @id
all_record_sets = list(dataset.record_sets)

if len(all_record_sets) == 0:
    print("No record sets found in the dataset.")
else:
    for rs in all_record_sets:
        print(f"\nRecord Set: @id = {rs.id}")
        if hasattr(rs, 'fields') and rs.fields:
            print("  Fields:")
            for f in rs.fields:
                print(f"    - {f.name} (@id: {f.id}) [dataType: {getattr(f, 'data_type', 'unknown') or 'unknown'}]")
        else:
            print("  (No fields defined)")


## 3. Data Extraction
We'll load records from each available record set into a pandas DataFrame.
Remember: always use the `@id` of the record set and fields when referencing elements.

In [ ]:
# Collect all record set @ids
record_sets = [rs.id for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_sets:
    print(f"Loading records for: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  Loaded {len(df)} records; columns:", df.columns.tolist())
    else:
        print("  No records found.")
# For demonstration, show the first table's columns and a preview
if dataframes:
    first_record_set_id = list(dataframes.keys())[0]
    print(f"\nPreview of data from record set @id: {first_record_set_id}")
    print(dataframes[first_record_set_id].head())
else:
    print("No tabular dataframes were created from the current record sets.")


## 4. Exploratory Data Analysis (EDA)
We will now try some simple analysis steps on the dataset.

***Note:*** This section assumes there is at least one record set with numeric fields present. Adjust the code to match the actual `@id`s of record sets and fields found in your overview (Section 2).

In [ ]:
# Choose a record set (by @id) and numeric field (by @id) for processing
if not dataframes:
    print("No dataframes to analyze.")
else:
    # Use the first available table and try to select a numeric field
    selected_record_set_id = list(dataframes.keys())[0]
    df = dataframes[selected_record_set_id]
    # Try to automatically select a numeric field by type or pattern
    numeric_field_id = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    if numeric_field_id is None:
        print(f"No numeric columns detected in record set {selected_record_set_id}.")
    else:
        print(f"Using record set: {selected_record_set_id}")
        print(f"Using numeric field (by @id): {numeric_field_id}")

        threshold = df[numeric_field_id].quantile(0.9) if len(df) > 0 else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        # Add a normalized column
        filtered_df = filtered_df.copy() # Avoid SettingWithCopyWarning
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to find a group field (categorical)
        group_field_id = None
        for col in df.columns:
            if col != numeric_field_id and df[col].dtype == object:
                group_field_id = col
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped results by {group_field_id}:")
            print(grouped_df.head())
        else:
            print("No categorical group field detected for grouping.")


## 5. Visualization
Let's create a visualization of the numeric field distribution and, if possible, the average value per group.

***Note:*** Adjust the plot as per your field types and actual data.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not dataframes:
    print("No data to visualize.")
elif numeric_field_id is None:
    print("No numeric field selected for visualization.")
else:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if 'group_field_id' in locals() and group_field_id:
        plt.figure(figsize=(8, 4))
        sns.barplot(
            data=df.groupby(group_field_id)[numeric_field_id].mean().reset_index(),
            x=group_field_id, y=numeric_field_id
        )
        plt.title(f"Average {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Average {numeric_field_id}")
        plt.xticks(rotation=45)
        plt.show()


## 6. Conclusion
We have loaded and explored the FAIR² dataset using `mlcroissant`, inspected its schema via Croissant `@id` references, and demonstrated basic data extraction and analysis workflows. Use the record set and field `@id`s for reproducible references to your dataset's structure during further analysis.